In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np 

/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
import pandas as pd 

import ast 
df = pd.read_excel("raw_agexport_db.xlsx")
df = df.filter(['category', 'partner_name', 'available_services'], axis = 1) 
df['available_services'] = df['available_services'].apply(ast.literal_eval)

In [7]:
all_available_services = {
    item.lower() 
    for sublist in df['available_services']
    for item in sublist
}

In [18]:
from groq import Groq
import json

groq_client = Groq() 

In [19]:
## extract knowledge from user message 
def service_description(groq_client: Groq, message: str) -> dict: 
    try: 
        response = groq_client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "system"
                        , "content": """
                        Eres un experto en estandarización de servicios de salud especializado en turismo médico internacional.
                        Tu tarea es generar definiciones claras, neutrales y estandarizadas de servicios de salud y bienestar ofrecidos por proveedores como dentistas, hospitales, clínicas, centros de rehabilitación y hoteles spa médicos.
                        Para cada nombre de servicio proporcionado:
                        Redacta una definición precisa de entre 150 y 250 carácteres.
                        Utiliza lenguaje formal, neutral y comprensible a nivel internacional.
                        Describe qué es el servicio, su finalidad clínica, terapéutica o estética, en qué consiste de manera general y a qué tipo de paciente o usuario está dirigido.
                        Haz una lista de los sintómas o consultas más comunes que alguien podría buscar con este servicio. 
                        Mantén terminología alineada con estándares internacionales del sector salud.
                        Si el nombre del servicio es ambiguo, utiliza la interpretación médica más aceptada.
                        Asegura coherencia en tono, estructura y nivel técnico en todas las definiciones generadas.
                        """
                    },
                    {
                        "role": "user",
                        "content": message
                    },
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "specialty_mapping",
                        "strict": True,
                        "schema": {
                            "type": "object",
                            "properties": {
                                "service": {"type": "string"},
                                "description": {"type": "string"}, 
                                "symptoms": {
                                    "type": "array",
                                    "items": {"type": "string"}
                                },
                            },
                            "required": ["service", "description", "symptoms"],
                            "additionalProperties": False
                        }
                    }
                }
        )
    
        result = json.loads(response.choices[0].message.content or "{}")
        # print(str(result))
        # flag = True 
    except: 
        result = {'service' : message, 'description' : None, 'symptoms' : None} 
        flag = False
    
    return result

In [ ]:
_result = service_description(groq_client, "radiología intervencionista y traumatología")
_result

{'service': 'Radiología intervencionista y traumatología',
 'description': 'La radiología intervencionista y la traumatología son especialidades que utilizan técnicas de imagen guiada para diagnosticar y tratar lesiones musculoesqueléticas, fracturas, hemorragias y patologías vasculares mediante procedimientos mínimamente invasivos.',
 'symptoms': ['dolor agudo en extremidades',
  'incapacidad para cargar peso',
  'deformidad ósea visible',
  'hematoma extensivo',
  'pérdida de movilidad articular',
  'dolor lumbar persistente',
  'sospecha de sangrado interno']}

In [20]:
all_available_services_explained = [] 
for _k in all_available_services: 
    
    if len(_k) > 0:
        _desc = service_description(
            groq_client= groq_client
            , message= str(_k)
        )

        if _desc['description'] is None: 
            print(f"!!!!!!!Error describing {_k}")
        else:
            print(f"{_k} succesfully described")
            all_available_services_explained.append({
                "original_description": _k
                , "description" : _desc 
            }) 

masaje prenatal succesfully described
ultrasonido doppler color succesfully described
servicios dentales odontológicos succesfully described
neurofisiología succesfully described
diagnóstico por imágenes succesfully described
optometría succesfully described
faciales succesfully described
spa succesfully described
tratamiento de prostata succesfully described
prótesis parciales removibles succesfully described
quiromasaje succesfully described
cirugía vascular succesfully described
ofrecer servicios médicos y quirúrgicos de cirugía bariátrica y de todas las ramas de la medicina succesfully described
clínica de nutrición succesfully described
prótesis totales succesfully described
atención especializada a niños y adultos mayores succesfully described
rellenos faciales succesfully described
aumento de busto succesfully described
servicios médicos de urología succesfully described
glaucoma succesfully described
descargas deportivas succesfully described
enfermedades de transmisión sexual 

In [ ]:
all_available_services_explained.append(all_available_services_explained)

In [14]:
all_available_services

{'',
 'abdominoplastía',
 'angiógrafo',
 'atención especializada a niños y adultos mayores',
 'audiometría',
 'aumento de busto',
 'baja visión',
 'banco de sangre',
 'biología molecular',
 'biopsia',
 'blanqueamiento de dientes',
 'blanqueamiento dental',
 'botox',
 'braquiplastía',
 'capacitaciones lesión cero',
 'cardiología',
 'cardiología invasiva y no invasiva',
 'carillas',
 'central de esterilización',
 'centro de la mujer',
 'centro del dolor y cuidados paliativos',
 'centro diabético',
 'centro para procedimientos ambulatorios',
 'circuito termal',
 'cirugia refractiva',
 'cirugía',
 'cirugía de busto',
 'cirugía de catarata',
 'cirugía de cejas',
 'cirugía de gluteos',
 'cirugía de mejillas',
 'cirugía de orejas',
 'cirugía de parpados',
 'cirugía de retina',
 'cirugía endocrina',
 'cirugía especializada de mano',
 'cirugía gastrointestinal',
 'cirugía general',
 'cirugía laparoscópica',
 'cirugía maxilofacial',
 'cirugía mínima invasiva',
 'cirugía oral',
 'cirugía plástica

In [13]:
all_available_services_explained[0]

{'original_description': 'traumatología y ortopedía',
 'description': {'service': 'Traumatología y ortopedia',
  'description': 'Especialidad que diagnostica y trata lesiones óseas, articulares y musculoesqueléticas, ofreciendo cirugías, rehabilitación y terapias avanzadas.',
  'symptoms': ['dolor articular',
   'limitación de movimiento',
   'inflamación y edema']}}

In [23]:
for index, _item in enumerate(all_available_services_explained):
    symp = ",".join(_item['description']['symptoms'])
    service_desc = f"{_item['description']['service']}: {_item['description']['description']} Atiende:{symp}"

    service_emb = model.encode(service_desc)

    _item['service_embedding'] = service_emb

    all_available_services_explained[index] = _item

In [1]:
import pickle

with open("services_explained.pkl", "wb") as f:   # wb = write binary
    pickle.dump(all_available_services_explained, f)


NameError: name 'all_available_services_explained' is not defined

In [3]:
import pickle
with open("services_explained.pkl", "rb") as f:
    data = pickle.load(f)

EOFError: Ran out of input

In [5]:
import numpy as np
from sklearn.decomposition import PCA

# data = all_available_services_explained

# Extract embeddings
embeddings = np.array([item['service_embedding'] for item in data])

# Extract service name and description
service_names = [item['description']['service'] for item in data]
descriptions = [item['description']['description'] for item in data]

# PCA to 2D
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)


In [7]:
import plotly.express as px
import pandas as pd

# Create DataFrame for plotting
df_plot = pd.DataFrame({
    "PC1": reduced[:, 0],
    "PC2": reduced[:, 1],
    "Service": service_names,
    "Description": descriptions
})

fig = px.scatter(
    df_plot,
    x="PC1",
    y="PC2",
    hover_data={
        "Service": True,
        "Description": True,
        "PC1": False,
        "PC2": False
    },
    title="PCA of Service Embeddings"
    , height = 800
)

fig.update_traces(marker=dict(size=10))
fig.show()


In [24]:
# --- UMAP + HDBSCAN clustering + Plotly visualization (best practice for embeddings) ---

import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
import plotly.express as px

# Install if needed (Jupyter-safe)
import umap
import hdbscan


# 1) Extract embeddings + metadata
embeddings = np.array([item["service_embedding"] for item in data])
service_names = [item["description"]["service"] for item in data]
descriptions = [item["description"]["description"] for item in data]

# 2) Normalize embeddings (important for cosine-based similarity)
X = normalize(embeddings)

# 3) Dimensionality reduction with UMAP (better than PCA for embeddings)
reducer = umap.UMAP(
    n_neighbors=15,        # local neighborhood size
    min_dist=0.1,          # how tightly points cluster
    n_components=2,
    metric="cosine",
    random_state=42
)

X_2d = reducer.fit_transform(X)

# 4) Cluster using HDBSCAN (natural pairing with UMAP)
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=max(5, int(len(X) * 0.03)),
    min_samples=None,
    metric="euclidean",   # use euclidean on UMAP output
    cluster_selection_method="eom"
)

clusters = clusterer.fit_predict(X_2d)
probs = clusterer.probabilities_

# 5) Build plotting dataframe
df_plot = pd.DataFrame({
    "UMAP1": X_2d[:, 0],
    "UMAP2": X_2d[:, 1],
    "Cluster": clusters,
    "ClusterLabel": np.where(clusters == -1, "Noise (-1)", clusters.astype(str)),
    "Service": service_names,
    "Description": descriptions,
    "ClusterProb": probs
})

n_noise = (clusters == -1).sum()
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)

print(f"Found {n_clusters} clusters")
print(f"Noise points: {n_noise}/{len(clusters)} ({n_noise/len(clusters):.1%})")

# 6) Plot interactive scatter
fig = px.scatter(
    df_plot,
    x="UMAP1",
    y="UMAP2",
    color="ClusterLabel",
    hover_name="Service",
    hover_data={
        "Description": True,
        "ClusterProb": True,
        "ClusterLabel": True,
        "UMAP1": False,
        "UMAP2": False
    },
    title=f"UMAP Projection of Service Embeddings with HDBSCAN Clustering "
          f"(Clusters={n_clusters}, Noise={n_noise})",
    height=900
)

fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.update_layout(legend_title_text="Cluster")

fig.show()


/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

/Users/javi.fong/Documents/wais/agexport/agexport-smart-directory/venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Found 2 clusters
Noise points: 0/235 (0.0%)
